# Customer Segmentation Using RFM Analysis

Analyze customer purchasing behavior using RFM (Recency, Frequency, Monetary) analysis to better understand customer value and segmentation.

**Business Gold**: Identify valuable customers, at-risk customers, and customer purchasing patterns to support targeted marketing and retention strategies.


## 1. Import Libraries

Import libraries for data analysis and visualization.

In [ ]:
import pandas as pd

df = pd.read_csv("online_retail.csv")

## 2. Load Dataset

Load customer transaction data and inspect the dataset structure.

In [ ]:
# Cleaning
df = df.dropna(subset=["CustomerID", "Description"])
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Sales"] = df["Quantity"] * df["UnitPrice"]

## 3. Data Cleaning

Clean missing values and remove invalid transactions before analysis.

In [ ]:
# Snapshot date
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

## 4. Feature Engineering

Create additional variables such as sales-related metrics.

In [ ]:
# RFM table
rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "InvoiceNo": "nunique",
    "Sales": "sum"
}).reset_index()

rfm.columns = ["CustomerID", "Recency", "Frequency", "Monetary"]

## 5. RFM Calculation

Calculate Recency, Frequency, and Monetary metrics for each customer.

In [ ]:
# Score each metric
rfm["R_score"] = pd.qcut(rfm["Recency"], 4, labels=[4, 3, 2, 1])
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4])
rfm["M_score"] = pd.qcut(rfm["Monetary"], 4, labels=[1, 2, 3, 4])

rfm["RFM_Score"] = (
    rfm["R_score"].astype(str) +
    rfm["F_score"].astype(str) +
    rfm["M_score"].astype(str)
)

## 6. RFM Scoring

Convert RFM metrics into customer scores for comparison.

In [ ]:
# Simple segment rule
def segment_customer(row):
    if row["RFM_Score"] in ["444", "443", "434", "344"]:
        return "VIP"
    elif row["R_score"] in [1, 2]:
        return "At Risk"
    else:
        return "Regular"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

## 7. Customer Segmentation

Segment customers based on purchasing behavior.

In [ ]:
print(rfm.head())

## 8. Segment Analysis & Insights

Analyze customer groups and summarize key business insights.

In [ ]:
print(rfm["Segment"].value_counts())

## 9. Business Recommendations

Provide actionable recommendations based on customer behavior.

## 10. Export CSV for Tableau Dashboard

Export segmentation results to CSV files for Tableau visualization.

In [ ]:
import os
os.makedirs("../outputs", exist_ok=True)

# Full RFM table with segment labels
rfm.to_csv("../outputs/rfm_segments.csv", index=False)

# Segment-level summary (count, avg RFM, total revenue)
segment_summary = rfm.groupby("Segment").agg(
    customer_count=("CustomerID", "count"),
    avg_recency=("Recency", "mean"),
    avg_frequency=("Frequency", "mean"),
    avg_monetary=("Monetary", "mean"),
    total_revenue=("Monetary", "sum")
).reset_index().round(2)
segment_summary.to_csv("../outputs/segment_summary.csv", index=False)

print("Saved:")
print("  ../outputs/rfm_segments.csv -", len(rfm), "customers")
print("  ../outputs/segment_summary.csv -", len(segment_summary), "segments")
print()
print(segment_summary)